# Corrected quantum transport simulation

**Important:** Run every cell from the beginning in a fresh kernel. The earlier notebook could mutate the global initial wavefunction during a test run, so the previously logged 192-run results must be treated as preliminary and regenerated. This corrected version regenerates the Gaussian initial state for every realization, uses periodic filtering consistent with the Fourier propagator, and reports spatial variance rather than calling it MSD.

The parameter called `smoothing_level` controls spatial correlation length; it is not an increasing complexity level. The composite landscape index is exploratory, not a validated universal measure of complexity. The validation section includes a free-particle analytical benchmark, spatial-grid refinement, and a nonzero-potential time-step refinement test. These checks do not replace a full convergence study for every landscape family.


In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.ndimage import gaussian_filter
from scipy.stats import linregress
from pathlib import Path
import time
import json
import os

print("Environment loaded.")
print("NumPy:", np.__version__)


In [ ]:
# Simulation parameters
N = 128
HBAR = 1.0
MASS = 1.0

L = 40.0
DX = L / N

DT = 0.01
N_STEPS = 500
SNAPSHOT_EVERY = 10

SIGMA = 2.0
X0 = -10.0
Y0 = 0.0
KX = 2.0
KY = 0.0

V_SCALE = 0.8

N_REALIZATIONS = 8
N_SMOOTHING_LEVELS = 8
BASE_SEED = 20260903

OUTPUT_DIR = Path("quantum_transport_results_corrected")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Grid:", N, "x", N)
print("DX:", DX)
print("DT:", DT)
print("Total simulated time:", DT * N_STEPS)


In [ ]:
# Coordinate and Fourier grids
x = np.linspace(-L / 2, L / 2, N, endpoint=False)
y = np.linspace(-L / 2, L / 2, N, endpoint=False)
X, Y = np.meshgrid(x, y, indexing="ij")

kx_grid = 2 * np.pi * np.fft.fftfreq(N, d=DX)
ky_grid = 2 * np.pi * np.fft.fftfreq(N, d=DX)
KX_GRID, KY_GRID = np.meshgrid(kx_grid, ky_grid, indexing="ij")
K2 = KX_GRID**2 + KY_GRID**2

# For T = p^2/(2m), the Fourier-space phase is exp[-i hbar k^2 dt/(2m)].
KINETIC_PHASE = np.exp(-1j * DT * HBAR * K2 / (2 * MASS))

print("Grid constructed.")


In [ ]:
# Normalized initial Gaussian wavepacket
def gaussian_wavepacket(X, Y, x0=X0, y0=Y0, sigma=SIGMA, kx=KX, ky=KY):
    psi = np.exp(-((X - x0)**2 + (Y - y0)**2) / (4 * sigma**2)).astype(complex)
    psi *= np.exp(1j * (kx * X + ky * Y))
    norm = np.sqrt(np.sum(np.abs(psi)**2) * DX * DX)
    if not np.isfinite(norm) or norm == 0:
        raise ValueError("Initial wavefunction has an invalid norm.")
    return psi / norm

# Immutable reference state. The propagator below always works on its own copy.
psi0 = gaussian_wavepacket(X, Y)
print("Initial normalization:", np.sum(np.abs(psi0)**2) * DX * DX)


In [ ]:
# Landscape descriptors
# These are operational descriptors of a generated field. Their weighted
# average is exploratory and should not be treated as a validated universal
# complexity measure.

def normalized_shannon_entropy(probabilities, n_categories=None):
    p = np.asarray(probabilities, dtype=float).ravel()
    p = np.where(np.isfinite(p) & (p > 0), p, 0.0)
    total = p.sum()
    if total <= 0:
        return 0.0
    p = p / total
    p = p[p > 0]
    categories = int(n_categories if n_categories is not None else len(probabilities))
    if categories <= 1:
        return 0.0
    H = -np.sum(p * np.log(p))
    return float(H / np.log(categories))

def spectral_entropy(V):
    power = np.abs(np.fft.fft2(V))**2
    power[0, 0] = 0.0
    return normalized_shannon_entropy(power, n_categories=power.size - 1)

def gradient_entropy(V, bins=64):
    gx = np.gradient(V, DX, axis=0)
    gy = np.gradient(V, DX, axis=1)
    magnitude = np.sqrt(gx**2 + gy**2)
    counts, _ = np.histogram(magnitude.ravel(), bins=bins, range=(0, np.max(magnitude) + 1e-12))
    return normalized_shannon_entropy(counts, n_categories=bins)

def multiscale_roughness(V, scales=(1, 2, 4, 8)):
    base_std = np.std(V)
    if base_std <= 0:
        return 0.0
    # Fractional residual variation at fixed, declared smoothing scales.
    fractions = []
    for sigma in scales:
        smooth = gaussian_filter(V, sigma=sigma, mode="wrap")
        fractions.append(np.std(V - smooth) / base_std)
    return float(np.mean(fractions))

def landscape_descriptors(V):
    se = spectral_entropy(V)
    ge = gradient_entropy(V)
    mr = multiscale_roughness(V)
    # Equal-weight summary is retained only for exploratory visualization.
    exploratory_index = (se + ge + mr) / 3.0
    return exploratory_index, {
        "spectral_entropy": se,
        "gradient_entropy": ge,
        "multiscale_roughness": mr
    }

# Backwards-compatible name used in a few plotting cells.
landscape_complexity = landscape_descriptors


In [ ]:
# Potential-landscape generators
# All filtering uses periodic wrapping to match the Fourier propagator's
# periodic boundary conditions.

def ordered_landscape():
    return V_SCALE * (0.5 + 0.5 * np.cos(2 * np.pi * X / 5.0))

def random_landscape(rng, correlation=0.0):
    noise = rng.normal(0.0, 1.0, size=(N, N))
    if correlation > 0:
        noise = gaussian_filter(noise, sigma=correlation, mode="wrap")
    noise -= noise.mean()
    std = np.std(noise)
    if std > 0:
        noise /= std
    return V_SCALE * noise

def multiscale_landscape(rng, smoothing_parameter):
    noise1 = rng.normal(0, 1, (N, N))
    noise2 = rng.normal(0, 1, (N, N))
    noise3 = rng.normal(0, 1, (N, N))

    scale_small = 0.5 + 5.0 * smoothing_parameter
    scale_medium = 1.0 + 10.0 * smoothing_parameter
    scale_large = 3.0 + 15.0 * smoothing_parameter

    component1 = gaussian_filter(noise1, sigma=max(0.5, scale_small), mode="wrap")
    component2 = gaussian_filter(noise2, sigma=max(1.0, scale_medium), mode="wrap")
    component3 = gaussian_filter(noise3, sigma=max(2.0, scale_large), mode="wrap")

    V = ((1.0 - smoothing_parameter) * component1
         + smoothing_parameter * component2
         + 0.7 * smoothing_parameter * component3)
    V -= V.mean()
    std = np.std(V)
    if std > 0:
        V /= std
    return V_SCALE * V

def binary_landscape(rng, smoothing_parameter):
    noise = rng.normal(0, 1, size=(N, N))
    sigma = 0.5 + 8.0 * smoothing_parameter
    smooth = gaussian_filter(noise, sigma=sigma, mode="wrap")
    threshold = np.percentile(smooth, 50)
    V = (smooth > threshold).astype(float)
    V -= V.mean()
    std = np.std(V)
    if std > 0:
        V /= std
    return V_SCALE * V


In [ ]:
# Landscape generator: level is a smoothing/correlation parameter, not complexity.
def generate_landscape(level, rng, family="multiscale"):
    if family == "ordered":
        return ordered_landscape()

    if not 0 <= level < N_SMOOTHING_LEVELS:
        raise ValueError("level must be between 0 and N_SMOOTHING_LEVELS - 1")

    smoothing_parameter = level / (N_SMOOTHING_LEVELS - 1)

    if family == "random":
        return random_landscape(rng, correlation=0.5 + 8.0 * smoothing_parameter)
    if family == "multiscale":
        return multiscale_landscape(rng, smoothing_parameter)
    if family == "binary":
        return binary_landscape(rng, smoothing_parameter)

    raise ValueError(f"Unknown landscape family: {family}")


In [ ]:
#CELL 8— Visualize environments BEFORE running anything

fig,axes=plt.subplots(
    2,
    4,
    figsize=(16,8)
)

rng=np.random.default_rng(BASE_SEED)

for level in range(N_SMOOTHING_LEVELS):

    V=generate_landscape(
        level,
        rng,
        family="multiscale"
    )

    C,components=landscape_complexity(V)

    ax=axes.flat[level]

    im=ax.imshow(
        V.T,
        origin="lower",
        extent=[
            x.min(),
            x.max(),
            y.min(),
            y.max()
        ],
        aspect="equal"
    )

    ax.set_title(
        f"Smoothing level {level}\nIndex={C:.3f}"
    )

    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR/"landscape_overview.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Split-step Fourier propagator (Strang splitting)
def evolve_wavefunction(psi, V, n_steps=N_STEPS, snapshot_every=SNAPSHOT_EVERY):
    psi = np.array(psi, dtype=complex, copy=True)  # Critical: never mutate caller's initial state.
    V = np.asarray(V, dtype=float)

    if psi.shape != (N, N) or V.shape != (N, N):
        raise ValueError("psi and V must both have shape (N, N).")
    if n_steps < 0 or snapshot_every <= 0:
        raise ValueError("n_steps must be nonnegative and snapshot_every positive.")

    potential_phase = np.exp(-1j * V * DT / (2 * HBAR))
    snapshots, times, norm_history = [], [], []

    for step in range(n_steps + 1):
        if step % snapshot_every == 0 or step == n_steps:
            snapshots.append(psi.copy())
            times.append(step * DT)
            norm_history.append(np.sum(np.abs(psi)**2) * DX * DX)

        if step == n_steps:
            break

        psi *= potential_phase
        psi_k = np.fft.fft2(psi)
        psi_k *= KINETIC_PHASE
        psi = np.fft.ifft2(psi_k)
        psi *= potential_phase

    return np.asarray(times), np.asarray(snapshots), np.asarray(norm_history)


In [ ]:
# Observables
def probability_density(psi):
    return np.abs(psi)**2

def expectation_x(psi):
    return float(np.sum(X * np.abs(psi)**2) * DX * DX)

def expectation_y(psi):
    return float(np.sum(Y * np.abs(psi)**2) * DX * DX)

def spatial_variance(psi):
    rho = np.abs(psi)**2
    x_mean = np.sum(X * rho) * DX * DX
    y_mean = np.sum(Y * rho) * DX * DX
    x2_mean = np.sum(X**2 * rho) * DX * DX
    y2_mean = np.sum(Y**2 * rho) * DX * DX
    return float((x2_mean - x_mean**2) + (y2_mean - y_mean**2))

def edge_probability(psi, edge_cells=4):
    rho = np.abs(psi)**2
    mask = np.zeros_like(rho, dtype=bool)
    mask[:edge_cells, :] = True
    mask[-edge_cells:, :] = True
    mask[:, :edge_cells] = True
    mask[:, -edge_cells:] = True
    return float(np.sum(rho[mask]) * DX * DX)

def calculate_ipr(psi):
    rho = np.abs(psi)**2
    return float(np.sum(rho**2) * DX * DX)

def calculate_participation_area(psi):
    ipr = calculate_ipr(psi)
    return np.inf if ipr <= 0 else 1.0 / ipr

def calculate_transport_observables(snapshots):
    variance, ipr, xmean, ymean, edge_mass = [], [], [], [], []
    for psi in snapshots:
        variance.append(spatial_variance(psi))
        ipr.append(calculate_ipr(psi))
        xmean.append(expectation_x(psi))
        ymean.append(expectation_y(psi))
        edge_mass.append(edge_probability(psi))
    return {
        "spatial_variance": np.asarray(variance),
        "ipr": np.asarray(ipr),
        "participation_area": 1.0 / np.asarray(ipr),
        "xmean": np.asarray(xmean),
        "ymean": np.asarray(ymean),
        "edge_probability": np.asarray(edge_mass)
    }

# Retain old function name for compatibility, but do not label this quantity MSD.
calculate_msd = spatial_variance


In [ ]:
# Single-run test. The global psi0 is not modified by this call.
rng = np.random.default_rng(BASE_SEED)
V = generate_landscape(4, rng, family="multiscale")
index_value, components = landscape_complexity(V)

print("Exploratory landscape index:", index_value)
print("Descriptor components:", components)

start = time.time()
times, snapshots, norms = evolve_wavefunction(psi0, V)
elapsed = time.time() - start

print("Simulation time:", elapsed, "seconds")
print("Initial norm:", norms[0])
print("Final norm:", norms[-1])
print("Maximum norm error:", np.max(np.abs(norms - norms[0])))
print("Global psi0 unchanged:", np.allclose(psi0, gaussian_wavepacket(X, Y)))


In [ ]:
#cell 12- plotting the actual wavepacket evolution FINALLY YAY KMSSS

fig,axes=plt.subplots(
    1,
    4,
    figsize=(18,4)
)

indices=np.linspace(
    0,
    len(snapshots)-1,
    4,
    dtype=int
)

for ax,index in zip(axes,indices):

    rho=np.abs(
        snapshots[index]
    )**2

    ax.imshow(
        rho.T,
        origin="lower",
        extent=[
            x.min(),
            x.max(),
            y.min(),
            y.max()
        ],
        aspect="equal"
    )

    ax.set_title(
        f"t={times[index]:.2f}"
    )

    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR/"wavepacket_evolution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#cell 13- plotting transport observables

observables=calculate_transport_observables(
    snapshots
)

fig,ax=plt.subplots(figsize=(8,5))

ax.plot(
    times,
    observables["spatial_variance"],
    linewidth=2
)

ax.set_xlabel("Time")
ax.set_ylabel("Spatial Variance")
ax.set_title("Quantum Wavepacket Spreading")

ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR/"spatial_variance_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


fig,ax=plt.subplots(figsize=(8,5))

ax.plot(
    times,
    observables["ipr"],
    linewidth=2
)

ax.set_xlabel("Time")
ax.set_ylabel("Inverse Participation Ratio")
ax.set_title("Inverse Participation Ratio")

ax.grid(alpha=0.25)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR/"ipr_test.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Reproducible ensemble experiment
def run_experiment(families=None, n_levels=N_SMOOTHING_LEVELS, n_realizations=N_REALIZATIONS):
    if families is None:
        families = ["random", "multiscale", "binary"]

    all_results = []
    experiment_start = time.time()
    total_runs = len(families) * n_levels * n_realizations
    run_number = 0

    for family_index, family in enumerate(families):
        print("\\n" + "=" * 60)
        print("LANDSCAPE FAMILY:", family)
        print("=" * 60)

        for level in range(n_levels):
            for realization in range(n_realizations):
                run_number += 1
                seed = BASE_SEED + 10000 * family_index + 100 * level + realization
                rng = np.random.default_rng(seed)

                V = generate_landscape(level, rng, family=family)
                index_value, components = landscape_complexity(V)

                # Regenerate the same analytical initial condition for every run.
                psi_initial = gaussian_wavepacket(X, Y)
                times, snapshots, norms = evolve_wavefunction(psi_initial, V)
                obs = calculate_transport_observables(snapshots)

                final_variance = obs["spatial_variance"][-1]
                final_ipr = obs["ipr"][-1]
                norm_error = np.max(np.abs(norms - norms[0]))

                for i, t in enumerate(times):
                    all_results.append({
                        "family": family,
                        "smoothing_level": level,
                        "realization": realization,
                        "seed": seed,
                        "exploratory_index": index_value,
                        **components,
                        "time": t,
                        "spatial_variance": obs["spatial_variance"][i],
                        "ipr": obs["ipr"][i],
                        "participation_area": obs["participation_area"][i],
                        "x_mean": obs["xmean"][i],
                        "y_mean": obs["ymean"][i],
                        "edge_probability": obs["edge_probability"][i],
                        "final_spatial_variance": final_variance,
                        "final_ipr": final_ipr,
                        "normalization_error": norm_error
                    })

                print(f"[{run_number}/{total_runs}] {family} | smoothing={level} | "
                      f"realization={realization} | index={index_value:.3f}")

    results = pd.DataFrame(all_results)
    print("\\nExperiment complete. Total runtime:",
          round(time.time() - experiment_start, 2), "seconds")
    return results


In [ ]:
# Full experiment: 3 families x 8 smoothing levels x 8 realizations = 192 runs
results = run_experiment(
    families=["random", "multiscale", "binary"],
    n_levels=N_SMOOTHING_LEVELS,
    n_realizations=N_REALIZATIONS
)


In [ ]:
# Save regenerated raw data
raw_path = OUTPUT_DIR / "quantum_transport_raw_data_corrected.csv"
results.to_csv(raw_path, index=False)
print("Saved:", raw_path)
print("Rows:", len(results))
print("Columns:", list(results.columns))


In [ ]:
# One row per independent landscape realization
summary = (
    results.groupby(["family", "smoothing_level", "realization"], as_index=False)
    .agg(
        exploratory_index=("exploratory_index", "first"),
        spectral_entropy=("spectral_entropy", "first"),
        gradient_entropy=("gradient_entropy", "first"),
        multiscale_roughness=("multiscale_roughness", "first"),
        final_spatial_variance=("final_spatial_variance", "first"),
        final_ipr=("final_ipr", "first"),
        normalization_error=("normalization_error", "first"),
        max_edge_probability=("edge_probability", "max")
    )
)
summary_path = OUTPUT_DIR / "experiment_summary_corrected.csv"
summary.to_csv(summary_path, index=False)
summary.head()


In [ ]:
# Transport outcomes by smoothing level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for family in summary["family"].unique():
    subset = summary[summary["family"] == family]
    grouped = subset.groupby("smoothing_level").agg(
        variance_mean=("final_spatial_variance", "mean"),
        variance_sd=("final_spatial_variance", "std"),
        ipr_mean=("final_ipr", "mean"),
        ipr_sd=("final_ipr", "std")
    ).reset_index()

    axes[0].errorbar(grouped["smoothing_level"], grouped["variance_mean"],
                     yerr=grouped["variance_sd"], marker="o", capsize=3, label=family)
    axes[1].errorbar(grouped["smoothing_level"], grouped["ipr_mean"],
                     yerr=grouped["ipr_sd"], marker="o", capsize=3, label=family)

axes[0].set(xlabel="Smoothing level (increasing spatial correlation)", ylabel="Final spatial variance",
            title="Wavepacket spatial variance")
axes[1].set(xlabel="Smoothing level (increasing spatial correlation)", ylabel="Final inverse participation ratio",
            title="Wavepacket inverse participation ratio")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "transport_by_smoothing_level.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Descriptive within-level associations.
# This avoids pooling the large between-level trend into a single naive correlation.
# With only 8 realizations per level, treat these slopes as exploratory; do not
# interpret their p-values as definitive evidence.
for family in summary["family"].unique():
    subset = summary[summary["family"] == family].copy()
    subset["index_within_level"] = (
        subset["exploratory_index"]
        - subset.groupby("smoothing_level")["exploratory_index"].transform("mean")
    )
    subset["variance_within_level"] = (
        subset["final_spatial_variance"]
        - subset.groupby("smoothing_level")["final_spatial_variance"].transform("mean")
    )
    subset["ipr_within_level"] = (
        subset["final_ipr"]
        - subset.groupby("smoothing_level")["final_ipr"].transform("mean")
    )

    print("\\n", family)
    for outcome in ["variance_within_level", "ipr_within_level"]:
        fit = linregress(subset["index_within_level"], subset[outcome])
        print(outcome, "within-level slope =", round(fit.slope, 6),
              "| r =", round(fit.rvalue, 4))


In [ ]:
# Compare the measured descriptor components across landscape families.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
component_names = ["spectral_entropy", "gradient_entropy", "multiscale_roughness"]
for ax, component in zip(axes, component_names):
    for family in summary["family"].unique():
        subset = summary[summary["family"] == family]
        grouped = subset.groupby("smoothing_level")[component].agg(["mean", "std"]).reset_index()
        ax.errorbar(grouped["smoothing_level"], grouped["mean"], yerr=grouped["std"],
                    marker="o", capsize=3, label=family)
    ax.set_title(component.replace("_", " ").title())
    ax.set_xlabel("Smoothing level")
    ax.grid(alpha=0.25)
axes[0].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "landscape_descriptors_by_smoothing.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#cell 21- representative figure

family="multiscale"
level=4
realization=0

seed=(
    BASE_SEED+
    10000*[
        "random",
        "multiscale",
        "binary"
    ].index(family)+
    100*level+
    realization
)

rng=np.random.default_rng(seed)

V=generate_landscape(
    level,
    rng,
    family=family
)

index_value,_=landscape_complexity(V)

times,snapshots,norms=evolve_wavefunction(
    gaussian_wavepacket(X,Y),
    V
)

fig,axes=plt.subplots(
    1,
    3,
    figsize=(15,5)
)

axes[0].imshow(
    V.T,
    origin="lower",
    extent=[
        x.min(),
        x.max(),
        y.min(),
        y.max()
    ],
    aspect="equal"
)

axes[0].set_title(
    f"Potential Landscape\nIndex={index_value:.3f}"
)

axes[0].set_xlabel("x")
axes[0].set_ylabel("y")


mid=len(snapshots)//2

axes[1].imshow(
    np.abs(snapshots[mid])**2,
    origin="lower",
    extent=[
        x.min(),
        x.max(),
        y.min(),
        y.max()
    ],
    aspect="equal"
)

axes[1].set_title(
    f"Probability Density\nt={times[mid]:.2f}"
)

axes[1].set_xlabel("x")
axes[1].set_ylabel("y")


axes[2].imshow(
    np.abs(snapshots[-1])**2,
    origin="lower",
    extent=[
        x.min(),
        x.max(),
        y.min(),
        y.max()
    ],
    aspect="equal"
)

axes[2].set_title(
    f"Probability Density\nt={times[-1]:.2f}"
)

axes[2].set_xlabel("x")
axes[2].set_ylabel("y")

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR/"representative_simulation.png",
    dpi=400,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Numerical diagnostics from the regenerated ensemble
print("=" * 60)
print("ENSEMBLE DIAGNOSTICS")
print("=" * 60)
print("Independent landscapes:", summary.shape[0])
print("Families:", sorted(summary["family"].unique()))
print("Smoothing levels:", summary["smoothing_level"].nunique())
print("Maximum norm drift:", results["normalization_error"].max())
print("Mean norm drift:", results["normalization_error"].mean())


In [ ]:
# Numerical validation: analytical free-particle benchmark, spatial refinement,
# and nonzero-potential time-step refinement.

def propagate_split_step(psi_initial, V, dt, total_time):
    n_steps = int(round(total_time / dt))
    if not np.isclose(n_steps * dt, total_time):
        raise ValueError("total_time must be an integer multiple of dt.")
    kinetic_phase = np.exp(-1j * dt * HBAR * K2 / (2 * MASS))
    potential_phase = np.exp(-1j * np.asarray(V) * dt / (2 * HBAR))
    psi = np.array(psi_initial, dtype=complex, copy=True)
    for _ in range(n_steps):
        psi *= potential_phase
        psi_k = np.fft.fft2(psi)
        psi_k *= kinetic_phase
        psi = np.fft.ifft2(psi_k)
        psi *= potential_phase
    return psi

def free_grid_benchmark(n_grid, dt=DT, total_time=2.0):
    dx = L / n_grid
    grid = np.linspace(-L / 2, L / 2, n_grid, endpoint=False)
    XX, YY = np.meshgrid(grid, grid, indexing="ij")
    psi = np.exp(-((XX-X0)**2 + (YY-Y0)**2)/(4*SIGMA**2)).astype(complex)
    psi *= np.exp(1j*(KX*XX + KY*YY))
    psi /= np.sqrt(np.sum(np.abs(psi)**2) * dx * dx)

    k = 2*np.pi*np.fft.fftfreq(n_grid, d=dx)
    KXg, KYg = np.meshgrid(k, k, indexing="ij")
    phase = np.exp(-1j*dt*HBAR*(KXg**2+KYg**2)/(2*MASS))

    for _ in range(int(round(total_time/dt))):
        psi = np.fft.ifft2(np.fft.fft2(psi) * phase)

    rho = np.abs(psi)**2
    norm = np.sum(rho)*dx*dx
    xmean = np.sum(XX*rho)*dx*dx/norm
    ymean = np.sum(YY*rho)*dx*dx/norm
    x2 = np.sum(XX**2*rho)*dx*dx/norm
    y2 = np.sum(YY**2*rho)*dx*dx/norm
    variance = (x2-xmean**2)+(y2-ymean**2)

    exact_x = X0 + HBAR*KX*total_time/MASS
    exact_y = Y0 + HBAR*KY*total_time/MASS
    exact_variance = 2*(SIGMA**2 + (HBAR*total_time/(2*MASS*SIGMA))**2)
    return norm, xmean, ymean, variance, exact_x, exact_y, exact_variance

print("FREE-PARTICLE SPATIAL GRID REFINEMENT")
for n_grid in [64, 128, 256]:
    values = free_grid_benchmark(n_grid)
    norm, xmean, ymean, variance, exact_x, exact_y, exact_variance = values
    print(f"N={n_grid}: norm={norm:.12f}, x error={abs(xmean-exact_x):.3e}, "
          f"y error={abs(ymean-exact_y):.3e}, variance error={abs(variance-exact_variance):.3e}")

# Time-step refinement must use a nonzero potential. For V=0, the Fourier
# kinetic propagator is exact at each step and the comparison is uninformative.
T_refine = 2.0
V_refine = generate_landscape(3, np.random.default_rng(BASE_SEED+777777), family="multiscale")
psi_refine = gaussian_wavepacket(X, Y)
psi_dt = propagate_split_step(psi_refine, V_refine, DT, T_refine)
psi_half_dt = propagate_split_step(psi_refine, V_refine, DT/2, T_refine)
l2_difference = np.sqrt(np.sum(np.abs(psi_dt-psi_half_dt)**2)*DX*DX)
print("NONZERO-POTENTIAL TIME-STEP REFINEMENT")
print("L2 difference between dt and dt/2 states:", l2_difference)
print("Norm at dt:", np.sum(np.abs(psi_dt)**2)*DX*DX)
print("Norm at dt/2:", np.sum(np.abs(psi_half_dt)**2)*DX*DX)


In [ ]:
#animations part 1

# ============================================================
# PREPARE A TEST SIMULATION FOR ANIMATION
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

# Make sure the wavepacket exists
psi0=gaussian_wavepacket(X,Y)

# Create a test potential
rng=np.random.default_rng(BASE_SEED)

V=generate_landscape(
    4,
    rng,
    family="multiscale"
)

# Calculate landscape complexity
index_value,components=landscape_complexity(V)

print("Exploratory landscape index:",index_value)

# Run the quantum simulation
print("Running test simulation...")

times,snapshots,norms=evolve_wavefunction(
    psi0.copy(),
    V,
    n_steps=N_STEPS,
    snapshot_every=SNAPSHOT_EVERY
)

print("Simulation complete!")
print("Number of animation frames:",len(snapshots))
print("Initial normalization:",norms[0])
print("Final normalization:",norms[-1])
print("Maximum normalization error:",np.max(np.abs(norms-norms[0])))

In [ ]:
# animation part 2

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML,display

fig,ax=plt.subplots(figsize=(8,7))

initial_density=np.abs(snapshots[0])**2

im=ax.imshow(
    initial_density.T,
    origin="lower",
    extent=[
        x.min(),
        x.max(),
        y.min(),
        y.max()
    ],
    aspect="equal"
)

ax.set_xlabel("x")
ax.set_ylabel("y")

title=ax.set_title(
    f"Quantum Wavepacket Evolution   t={times[0]:.2f}"
)

def update(frame):

    density=np.abs(
        snapshots[frame]
    )**2

    im.set_data(
        density.T
    )

    title.set_text(
        f"Quantum Wavepacket Evolution   t={times[frame]:.2f}"
    )

    return im,title


animation=FuncAnimation(
    fig,
    update,
    frames=len(snapshots),
    interval=50,
    blit=True
)

plt.close(fig)

display(
    HTML(
        animation.to_jshtml()
    )
)

In [ ]:
print("snapshots:",snapshots.shape)
print("times:",times)
print("psi0 exists:","psi0" in globals())
print("V exists:","V" in globals())
print("x exists:","x" in globals())
print("y exists:","y" in globals())

print("First frame max:",np.max(np.abs(snapshots[0])**2))
print("Last frame max:",np.max(np.abs(snapshots[-1])**2))
print("Frames identical:",np.allclose(snapshots[0],snapshots[-1]))

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(16,4))

indices=[0,len(snapshots)//3,2*len(snapshots)//3,len(snapshots)-1]

for ax,index in zip(axes,indices):
    density=np.abs(snapshots[index])**2

    ax.imshow(
        density.T,
        origin="lower",
        extent=[x.min(),x.max(),y.min(),y.max()],
        aspect="equal"
    )

    ax.set_title(f"t={times[index]:.2f}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.tight_layout()
plt.show()